In [ ]:
import rasterio
from rasterio.merge import merge
from rasterio.plot import show
import matplotlib.pyplot as plt
import glob
import os
import geopandas as gpd

from shapely.geometry import Polygon
from shapely.geometry import mapping

from atlite.gis import ExclusionContainer, shape_availability

In [ ]:
tiff_paths = [
    snakemake.input.slope_max,
    snakemake.input.airports,
    snakemake.input.glacier,
    snakemake.input.hydrology,
    snakemake.input.military,
    snakemake.input.powerlines,
    snakemake.input.radar,
    snakemake.input.rail_narrow,
    snakemake.input.rail_rail,
    snakemake.input.road_motorways_motor,
    snakemake.input.road_motorways_trunk,
    snakemake.input.road_motorways_link,
    snakemake.input.road_primary,
    snakemake.input.bufferedPipes,
    snakemake.input.bufferedFloat,
    snakemake.input.bufferedFixed,
    snakemake.input.shipping_high,
    snakemake.input.shipping_med,
    snakemake.input.luisaPort,
]

In [ ]:
src_files_to_mosaic = []

for fp in tiff_paths:
    src = rasterio.open(fp)
    src_files_to_mosaic.append(src)
    
# Merge the raster files
mosaic, out_transform = merge(src_files_to_mosaic)

# Close the file connections
for src in src_files_to_mosaic:
    src.close()

# Define the output file path
output_file = snakemake.output.techExcl

# Copy the metadata from one of the original files
out_meta = src.meta.copy()

# Update the metadata to match the merged data
out_meta.update({
    'driver': 'GTiff',
    'height': mosaic.shape[1],
    'width': mosaic.shape[2],
    'transform': out_transform,
    'count': mosaic.shape[0],
    'compress': 'lzw'  # Adding LZW compression
})

# Write the merged file
with rasterio.open(output_file, "w", **out_meta) as dest:
    dest.write(mosaic)
